HWモデル+BSモデルでの株価SDEを考える
```math
dS_t = \mu S_t dt + \sigma S_t dW^{S} \\
dB_t = r_t B_t dt \\
dr_t = \left( \theta(t) - \alpha(t) r_t \right) + \sigma_t dW^{r} \\
dW^{S} dW^{r} = \rho dt
```
$B_t$を基準材とするQリスク中立測度でのSDEを考える
```math
dS_t = r_t S_t dt + \sigma S_t dW^{S,Q} \\
dB_t = r_t B_t dt \\
dr_t = \left( \theta(t) - \alpha(t) r_t \right) + \sigma_t dW^{r,Q} \\
dW^{S} dW^{r} = \rho dt
```
それぞれ解析解を考える。
```math
DF(t,T) = \frac{B_t}{B_T} = \exp \left( - \int_{t}^{T} r_s ds \right)\\
S_T = S_t \exp \left( \int_{t}^{T} r_s ds - \frac{1}{2} \int_{t}^{T} \sigma^2 ds + \sigma \int_{t}^{T} dW^{S,Q} \right) \\
B_T = B_t \exp \left( \int_{t}^{T} r_s ds \right)
```
$T$セトル$K$執行のフォワード取引のPV
```math
Fwd(t,T) = \text{E}^{Q} \left[ DF(t,T) (S_T - K) \right]
```
$T$セトル$K$ストライクのコールオプションのPV
```math
Call(t,K,T) = \text{E}^Q \left[ DF(t,T) \text{max}(S_T - K, 0) \right]
```



OISスワップレートのセットからディスカウントカーブを取得する
```math
\text{OIS Swap Rate} : S_i \\
\text{Swap Tenor} : [0, T_i] \\
\text{Payment Frequency} : 6M = 0.5 \\
\text{Cashflow Period} : [0, 0.5],...,[T^{j-1}, T^{j}] , ... ,[T_i - 0.5 ,T_i] \\
\text{Payment Delay} : 2D = 2/365 \\
\\
\text{Floating Side of } j : \text{Notional} \left[ \prod_k (1+ON_k \delta_k) - 1 \right] = \text{Notional} \left[ \prod_k \frac{DF(t_{k})}{DF(t_{k+1})} - 1 \right] \\
\text{Fixed Side of } j : \text{Notional} \left[ S_i \sum_k \delta_k \right]
```
パースワップレートで満たされるのは
```math
0 = \text{E}^Q \left[ \sum_j DF(T_j + 2D) \left[ \prod_k \frac{DF(t_{k})}{DF(t_{k+1})} - 1 - S_i \sum \delta_k \right] \right]
```
フォワードレート$-\frac{d}{dT}log(DF(T))$を補間関数として$DF(T)$を計算する関数をアウトプットとする。  
補間関数のノードとして持つのは$-\frac{d}{dT}log(DF(T_j + 2D))$
```math
DF(t) = \exp\left( \int_{0}^{t} -\frac{d}{dT} log(DF(T)) dT \right)
```

OISスワップレートのセットからディスカウントカーブを取得する
```math
\text{OIS Swap Rate} : S_i \\
\text{Swap Tenor} : [0, T_i] \\
\text{Payment Frequency} : 6M = 0.5 \\
\text{Cashflow Period} : [0, 0.5],...,[T^{j-1}, T^{j}] , ... ,[T_i - 0.5 ,T_i] \\
\text{Payment Delay} : 2D = 2/365 \\
\\
\text{Floating Side of } j : \text{Notional} \left[ \prod_k (1+ON_k \delta_k) - 1 \right] = \text{Notional} \left[ \prod_k \frac{DF(t_{k})}{DF(t_{k+1})} - 1 \right] \\
\text{Fixed Side of } j : \text{Notional} \left[ S_i \sum_k \delta_k \right]
```
パースワップレートで満たされるのは
```math
0 = \text{E}^Q \left[ \sum_j DF(T_j + 2D) \left[ \prod_k \frac{DF(t_{k})}{DF(t_{k+1})} - 1 - S_i \sum \delta_k \right] \right]
```
$log(DF(T)$を補間関数として$DF(T)$を計算する関数をアウトプットとする。  
補間関数のノードとして持つのは$log(DF(T_j + 2D)$である。

In [4]:
import numpy as np
from scipy.optimize import brentq
from scipy.interpolate import CubicSpline, UnivariateSpline
from scipy.integrate import cumulative_trapezoid

# -----------------------------
# Log-linear DF interpolator
# -----------------------------
class LogDFCurve:
    def __init__(self):
        self.nodes = []      # times
        self.logdfs = []     # log DF

    def add_node(self, t, logdf):
        self.nodes.append(t)
        self.logdfs.append(logdf)
    
    def pop_node(self):
        self.nodes.pop()
        self.logdfs.pop()

    def df(self, t):
        t_arr = np.asarray(t, dtype=float)
        scalar_input = (t_arr.ndim == 0)

        t_flat = t_arr.reshape(-1)
        res = np.empty_like(t_flat)

        nodes = np.asarray(self.nodes)
        logdfs = np.asarray(self.logdfs)

        # t <= first node
        mask = t_flat <= nodes[0]
        res[mask] = np.exp(logdfs[0])

        # interpolation intervals
        for i in range(len(nodes) - 1):
            t0, t1 = nodes[i], nodes[i + 1]
            mask = (t_flat > t0) & (t_flat <= t1)
            if np.any(mask):
                w = (t_flat[mask] - t0) / (t1 - t0)
                logdf = (1 - w) * logdfs[i] + w * logdfs[i + 1]
                res[mask] = np.exp(logdf)

        # flat extrapolation (t > last node)
        mask = t_flat > nodes[-1]
        res[mask] = np.exp(logdfs[-1])

        res = res.reshape(t_arr.shape)
        return res.item() if scalar_input else res
    


class LogDFCurveSpline:
    def __init__(self):
        self.nodes = []      # times
        self.logdfs = []     # log DF
        self.spline = None

    def add_node(self, t, logdf):
        self.nodes.append(t)
        self.logdfs.append(logdf)
        if len(self.nodes) >= 2:
            self.spline = CubicSpline(self.nodes, self.logdfs, bc_type='natural', extrapolate=True)
    
    def pop_node(self):
        self.nodes.pop()
        self.logdfs.pop()
        if len(self.nodes) >= 2:
            self.spline = CubicSpline(self.nodes, self.logdfs, bc_type='natural', extrapolate=True)
        else:
            self.spline = None

    def df(self, t):
        t_arr = np.asarray(t, dtype=float)
        logdf = self.spline(t_arr)
        return np.exp(logdf)
    
    def forward_rate(self, t):
        t_arr = np.asarray(t, dtype=float)
        logdf = self.spline(t_arr)
        dlogdf_dt = self.spline.derivative(1)(t_arr)
        return -dlogdf_dt
    


    

class FwdRateCurveSpline:
    def __init__(self):
        self.nodes = []      # times
        self.logdfs = []     # log DF nodes
        self.spline = None   # spline on forward rates

    # --------------------------------
    # internal rebuild
    # --------------------------------
    def _rebuild_spline(self):
        if len(self.nodes) < 2:
            self.spline = None
            return

        t = np.asarray(self.nodes, dtype=float)
        logdf = np.asarray(self.logdfs, dtype=float)

        # forward rate nodes : f = - d logDF / dt
        fwdrates = -np.gradient(logdf, t)

        # spline on forward rates
        self.spline = CubicSpline(
            t, fwdrates,
            bc_type='natural',
            extrapolate=True
        )

    # --------------------------------
    # interface
    # --------------------------------
    def add_node(self, t, logdf):
        self.nodes.append(t)
        self.logdfs.append(logdf)
        self._rebuild_spline()

    def pop_node(self):
        self.nodes.pop()
        self.logdfs.pop()
        self._rebuild_spline()

    # --------------------------------
    # DF from fwdrate spline
    # --------------------------------
    def df(self, t):
        if self.spline is None:
            raise ValueError("Not enough nodes to build spline")

        t_arr = np.asarray(t, dtype=float)
        scalar_input = (t_arr.ndim == 0)

        t_flat = t_arr.reshape(-1)

        # integration grid = nodes + query points
        grid = np.unique(np.concatenate([self.nodes, t_flat]))
        grid.sort()

        # forward rates on grid
        f = self.spline(grid)

        # integrate f to get logDF
        logdf_grid = -cumulative_trapezoid(f, grid, initial=0.0)

        # anchor at t=0 (logDF(0)=0)
        # shift so that first node matches stored logdf
        idx0 = np.where(grid == self.nodes[0])[0][0]
        shift = self.logdfs[0] - logdf_grid[idx0]
        logdf_grid += shift

        # interpolate logDF
        logdf_interp = np.interp(t_flat, grid, logdf_grid)
        df = np.exp(logdf_interp)

        if scalar_input:
            return float(df[0])
        return df.reshape(t_arr.shape)

    # --------------------------------
    # forward rate = spline value
    # --------------------------------
    def forward_rate(self, t):
        if self.spline is None:
            raise ValueError("Not enough nodes to build spline")

        t_arr = np.asarray(t, dtype=float)
        f = self.spline(t_arr)
        return f
    



def make_ois_cashflows(T, pay_freq, delay, is_bullet):
    if is_bullet:
        return [{
            "t0": 0.0,
            "t1": T,
            "pay": T + delay,
            "accrual": T
        }]
    else:
        pay_times = np.arange(pay_freq, T + 1e-12, pay_freq)
        cfs = []
        for t in pay_times:
            cfs.append({
                "t0": t - pay_freq,
                "t1": t,
                "pay": t + delay,
                "accrual": pay_freq
            })
        return cfs



# -----------------------------
# OIS bootstrap
# -----------------------------
def bootstrap_ois(
    swap_maturities,
    swap_rates,
    is_bullet_flags,   # ★追加
    pay_freq=0.5,
    delay=2/365
):
    curve = FwdRateCurveSpline()
    curve.add_node(0.0, 0.0)

    for T, S, is_bullet in zip(swap_maturities, swap_rates, is_bullet_flags):

        cfs = make_ois_cashflows(T, pay_freq, delay, is_bullet)

        def pv_equation(logdf_T):
            curve.add_node(T + delay, logdf_T)

            pv_float = 0.0
            pv_fixed = 0.0

            for cf in cfs:
                df0 = curve.df(cf["t0"])
                df1 = curve.df(cf["t1"])
                dfp = curve.df(cf["pay"])

                pv_float += dfp * (df0 / df1 - 1.0)
                pv_fixed += dfp * cf["accrual"]

            curve.pop_node()
            return pv_float - S * pv_fixed

        logdf = brentq(
            pv_equation,
            a=-2.0 * T,
            b=0.1,
            maxiter=100
        )

        curve.add_node(T + delay, logdf)

    return curve



In [5]:
swap_maturities = [
    1/365, 7/365, 14/365, 21/365,
    1/12, 2/12, 3/12, 4/12, 5/12, 6/12,
    7/12, 8/12, 9/12, 10/12, 11/12,
    1, 15/12, 18/12,
    2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 12, 15, 20, 25, 30, 35, 40
]

swap_rates = [
    0.0072700, 0.0072760, 0.0072826, 0.0072938,
    0.0072893, 0.0072939, 0.0074357, 0.0076963, 0.0079437, 0.0081963,
    0.0084333, 0.0086607, 0.0088874, 0.0091402, 0.0093676,
    0.0096020, 0.0102250, 0.0108500,
    0.0120688, 0.0138000, 0.0150812, 0.0161250, 0.0170500,
    0.0179500, 0.0187950, 0.0196250, 0.0204660,
    0.0212625, 0.0220375, 0.0241867, 0.0271140,
    0.0289113, 0.0298972, 0.0304875, 0.0309025
]

is_bullet = [
    True, True, True, True, True, True,
    True, True, True, True, True, True,
    True, True, True, True, True,
    False, True, True,
    True, True, True, True, True, True, True, True, True, 
    True, True, True, True, True, True, True, True
]

curve = bootstrap_ois(swap_maturities, swap_rates, is_bullet)

print(curve.df(3.0))
print(curve.df(7.0))
print(curve.df([1,2,3,4]))

0.9600312724661054
0.8882108281760402
[0.99048292 0.97630829 0.96003026 0.94288294]


2026/1/19 JSCC

| Tenor | JPY OIS (%) |
| ----: | ----------: |
|    1D |     0.72700 |
|    1W |     0.72760 |
|    2W |     0.72826 |
|    3W |     0.72938 |
|    1M |     0.72893 |
|    2M |     0.72939 |
|    3M |     0.74357 |
|    4M |     0.76963 |
|    5M |     0.79437 |
|    6M |     0.81963 |
|    7M |     0.84333 |
|    8M |     0.86607 |
|    9M |     0.88874 |
|   10M |     0.91402 |
|   11M |     0.93676 |
|    1Y |     0.96020 |
|   15M |     1.02250 |
|   18M |     1.08500 |
|    2Y |     1.20688 |
|    3Y |     1.38000 |
|    4Y |     1.50812 |
|    5Y |     1.61250 |
|    6Y |     1.70500 |
|    7Y |     1.79500 |
|    8Y |     1.87950 |
|    9Y |     1.96250 |
|   10Y |     2.04660 |
|   11Y |     2.12625 |
|   12Y |     2.20375 |
|   15Y |     2.41867 |
|   20Y |     2.71140 |
|   25Y |     2.89113 |
|   30Y |     2.98972 |
|   35Y |     3.04875 |
|   40Y |     3.09025 |


In [6]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display




def update_plot(*args):
    try:
        swap_maturities, swap_rates, is_bullet = swap_table.get_data()
        swap_maturities = list(swap_maturities)
        swap_rates   = list(swap_rates)
        is_bullet_flags = list(is_bullet)
        if len(swap_maturities) < 2:
            return
    except:
        return


    curve = bootstrap_ois(swap_maturities, swap_rates, is_bullet_flags)

    grid = np.linspace(0.0, max(swap_maturities), 300)

    df_vals = [curve.df(t) for t in grid]
    fwd_rate_vals = [curve.forward_rate(t) for t in grid]

    with fig.batch_update():
        fig.data[0].x = grid
        fig.data[0].y = df_vals

        fig.data[1].x = grid
        fig.data[1].y = fwd_rate_vals

        fig.data[2].x = grid
        fig.data[2].y = df_vals


class DFTable:
    def __init__(self, swap_maturities, swap_rates, is_bullet):
        self.rows = []
        for t, r, b in zip(swap_maturities, swap_rates, is_bullet):
            self.add_row(t, r, b)

    def add_row(self, t=30.0, r=0.1, b=False):
        time = widgets.FloatText(value=t, step=0.1, layout=widgets.Layout(width="100px"))
        rate   = widgets.FloatText(value=r, step=0.001, layout=widgets.Layout(width="100px"))
        bullet = widgets.Checkbox(value=b, layout=widgets.Layout(width="40px"))
        btn  = widgets.Button(description="❌", layout=widgets.Layout(width="40px"))

        row = widgets.HBox([time, rate, bullet, btn])
        self.rows.append((time, rate, bullet, btn, row))

        btn.on_click(lambda _: self.remove_row(row))
        time.observe(update_plot, names="value")
        rate.observe(update_plot, names="value")
        bullet.observe(update_plot, names="value")

        self.render()
        update_plot()

    def remove_row(self, row_widget):
        self.rows = [r for r in self.rows if r[4] is not row_widget]
        self.render()
        update_plot()

    def get_data(self):
        swap_maturities = [r[0].value for r in self.rows]
        swap_rates   = [r[1].value for r in self.rows]
        is_bullet_flags = [r[2].value for r in self.rows]
        return zip(*sorted(zip(swap_maturities, swap_rates, is_bullet_flags)))

    def render(self):
        table.children = [header] + [r[4] for r in self.rows]

init_swap_maturities = [
    1/365, 7/365, 14/365, 21/365,
    1/12, 2/12, 3/12, 4/12, 5/12, 6/12,
    7/12, 8/12, 9/12, 10/12, 11/12,
    1, 15/12, 18/12,
    2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 12, 15, 20, 25, 30, 35, 40
]

init_swap_rates = [
    0.0072700, 0.0072760, 0.0072826, 0.0072938,
    0.0072893, 0.0072939, 0.0074357, 0.0076963, 0.0079437, 0.0081963,
    0.0084333, 0.0086607, 0.0088874, 0.0091402, 0.0093676,
    0.0096020, 0.0102250, 0.0108500,
    0.0120688, 0.0138000, 0.0150812, 0.0161250, 0.0170500,
    0.0179500, 0.0187950, 0.0196250, 0.0204660,
    0.0212625, 0.0220375, 0.0241867, 0.0271140,
    0.0289113, 0.0298972, 0.0304875, 0.0309025
]

init_is_bullet = [
    True, True, True, True, True, True,
    True, True, True, True, True, True,
    True, True, True, True, True,
    False, True, True,
    True, True, True, True, True, True, True, True, True, 
    True, True, True, True, True, True, True, True
]


header = widgets.HBox([
    widgets.Label("Swap Maturity", layout=widgets.Layout(width="100px")),
    widgets.Label("Swap Rate",   layout=widgets.Layout(width="100px")),
    widgets.Label("Is Bullet",   layout=widgets.Layout(width="40px")),
    widgets.Label("",     layout=widgets.Layout(width="40px"))
])

table = widgets.VBox()
swap_table = DFTable(init_swap_maturities, init_swap_rates, init_is_bullet)

add_button = widgets.Button(description="➕ Add Node")
add_button.on_click(lambda _: swap_table.add_row())

fig = go.FigureWidget(
    make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=["Discount Factor DF(0,t)", "Forward Rate", "Discount Factor DF(0,t)"]
    )
)

fig.add_trace(go.Scatter(name="Discount Factor DF(0,t)"), row=1, col=1)
fig.add_trace(go.Scatter(name="Forward Rate"),  row=2, col=1)
fig.add_trace(go.Scatter(name="Discount Factor DF(0,t)"), row=3, col=1)

fig.update_layout(height=700, showlegend=False)



controls = widgets.VBox([add_button])

display(widgets.HBox([table, controls]))
display(fig)

update_plot()


FigureWidget({
    'data': [{'name': 'Discount Factor DF(0,t)',
              'type': 'scatter',
             …